# 04 - Call signals in the Feature Store, and a survival model for churn

**Business question:** which accounts are likely to cancel in the next nine months, and do the topics customers raise on
calls tell us something product usage does not?

| Section | What it shows |
|---|---|
| 1 | Feature Store: entity `ACCOUNT`, a static snapshot calendar, two managed feature views computed from raw tables |
| 2 | A point-in-time training Dataset as of the snapshot date, with time-to-churn labels |
| 3 | Six models in a warehouse stored procedure: logistic vs Cox vs gradient-boosted survival, each usage-only and with call features |
| 4 | Experiment Tracking: one run per model, the c-index side by side |
| 5 | The survival models in the Model Registry, with a per-feature `explain` |
| 6 | Scoring from SQL at four as-of dates through the Feature Store, and a model monitor |

lifelines and scikit-survival come from the Snowflake package channel, so training runs next to the data. Runs standalone
after `03`. Every section ends in a `CHECK`.

In [ ]:
NOTEBOOK_TAG = "transcript-churn-04"

# Session: Snowsight Workspaces supplies one; a local kernel (Cortex Code Desktop,
# ./demo.sh build) opens the connection saved in .env by ./demo.sh setup.
import json
import os
import time

import numpy as np
import pandas as pd

DB = "TRANSCRIPT_CHURN_ML_DEMO"
WH = "TRANSCRIPT_CHURN_ML_WH"
ROLE = "SYSADMIN"
EMBED_MODEL = "snowflake-arctic-embed-l-v2.0"
LLM = "claude-sonnet-4-5"
# The account this demo is allowed to build in. ./demo.sh setup writes it to .env;
# in Workspaces, replace the placeholder with your demo account name.
DEMO_ACCOUNT = os.environ.get("DEMO_EXPECTED_ACCOUNT", "<YOUR_DEMO_ACCOUNT>")

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    LOCAL = False
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.config(
        "connection_name", os.environ.get("SNOWFLAKE_DEFAULT_CONNECTION_NAME", "default")).create()
    LOCAL = True

session.use_role(ROLE)
session.use_warehouse(WH)
session.use_database(DB)
session.sql(f"ALTER SESSION SET QUERY_TAG = '{NOTEBOOK_TAG}'").collect()

if LOCAL:
    # Local kernels have no SQL cells: run each %%sql cell through this session instead.
    from IPython.core.magic import register_cell_magic
    from IPython.display import display

    @register_cell_magic
    def sql(line, cell):
        args = line.split()
        name = args[args.index("-r") + 1] if "-r" in args else None
        rows = session.sql(cell).collect()      # collect(): CALL and EXECUTE cannot be described first
        result = pd.DataFrame([r.as_dict() for r in rows])
        if name:
            get_ipython().user_ns[name] = result
        display(result.head(50))

CHECKS = {}


def check(section, ok, evidence):
    status = "PASS" if ok else "FAIL"
    CHECKS[section] = (status, evidence)
    print(f"CHECK {section}: {status} | {evidence}")
    if not ok:
        raise AssertionError(f"CHECK {section} failed: fix the cause and re-run this section")


def q(text):
    return session.sql(text).to_pandas()


def one(text):
    return session.sql(text).collect()[0][0]


account, role = session.sql("SELECT CURRENT_ACCOUNT_NAME(), CURRENT_ROLE()").collect()[0]
if account != DEMO_ACCOUNT:
    raise SystemExit(f"STOP: connected to {account}, but this demo may only build in {DEMO_ACCOUNT}. "
                     "Set DEMO_EXPECTED_ACCOUNT (./demo.sh setup does) or edit DEMO_ACCOUNT in this cell.")
if not session.sql(f"SHOW SCHEMAS LIKE 'RAW' IN DATABASE {DB}").collect():
    raise SystemExit(f"STOP: {DB}.RAW is missing. Run lab/setup.sql (./demo.sh setup) first.")
print(f"Connected as role {role} on {account}, warehouse {WH}, query tag {NOTEBOOK_TAG} "
      f"({'local kernel' if LOCAL else 'Workspaces session'})")

In [ ]:
from snowflake.ml.feature_store import CreationMode, Entity, FeatureStore, FeatureView

if one(f"SELECT COUNT(*) FROM {DB}.ANALYTICS.PRICE_SENTIMENT") == 0:
    raise SystemExit("STOP: ANALYTICS.PRICE_SENTIMENT is empty. Run notebooks 01-03 first.")
SNAPSHOT = "2026-01-01"                 # training as-of date; outcomes are observed for 270 days after it
HORIZON_DAYS = 270
USAGE_FEATURES = ["LOGINS_L3M", "LOGIN_TREND", "REPORTS_L3M", "ACTIVE_USERS", "TENURE_DAYS", "MRR", "IS_TEAM"]
CALL_FEATURES = ["CALLS_90D", "SEGMENTS_90D", "PRICE_NEG_90D", "PRICE_COMPETITOR_90D", "PRICE_MODEL_90D",
                 "PRICE_VALUE_90D", "DISCOVERED_SEGMENTS_90D", "RENEWAL_SEGMENTS_90D"]
ALL_FEATURES = USAGE_FEATURES + CALL_FEATURES
# Re-runnable: drop what this notebook creates
for stmt in [f"DROP MODEL MONITOR IF EXISTS {DB}.ML.CHURN_MONITOR", f"DROP MODEL IF EXISTS {DB}.ML.CHURN_SURVIVAL",
             f"DROP DATASET IF EXISTS {DB}.FEATURES.CHURN_TRAINING", f"DROP TABLE IF EXISTS {DB}.FEATURES.CHURN_TRAINING_SOURCE"]:
    session.sql(stmt).collect()
fs = FeatureStore(session=session, database=DB, name="FEATURES", default_warehouse=WH,
                  creation_mode=CreationMode.CREATE_IF_NOT_EXIST)
print(f"Feature Store {DB}.FEATURES; snapshot {SNAPSHOT}; {len(USAGE_FEATURES)} usage + {len(CALL_FEATURES)} call features")

---
## 1. Features computed from the raw tables

One entity, `ACCOUNT`. Two **managed** feature views (dynamic tables) compute features for every account at every date in
`FEATURES.SNAPSHOT_CALENDAR`, from the raw usage events and from the call topics and price sentiment built in `01`-`03`:

- `ACCOUNT_USAGE_FV`: logins and reports over the last three months, the trend against the three months before, seats,
  tenure, plan value.
- `ACCOUNT_CALL_SIGNALS_FV`: calls, segments, negative price remarks by driver, segments in discovered topics, and renewal
  talk in the 90 days before each date.

The calendar is a plain table, so the views are deterministic (no `CURRENT_DATE()` inside them). Training and scoring read
the same definitions at different dates.

In [ ]:
%%sql -r snapshot_calendar
CREATE OR REPLACE TABLE TRANSCRIPT_CHURN_ML_DEMO.FEATURES.SNAPSHOT_CALENDAR COMMENT = 'As-of dates the feature views compute' AS
SELECT AS_OF::TIMESTAMP_NTZ AS AS_OF FROM VALUES ('2025-10-01'), ('2025-11-01'), ('2025-12-01'), ('2026-01-01') AS C(AS_OF)

In [ ]:
existing = [r["NAME"] for r in fs.list_entities().collect()]
if "ACCOUNT" not in existing:
    fs.register_entity(Entity(name="ACCOUNT", join_keys=["ACCOUNT_ID"], desc="A customer account"))
account = fs.get_entity("ACCOUNT")

usage_df = session.sql(f"""
WITH agg AS (
  SELECT a.ACCOUNT_ID, c.AS_OF, a.SIGNUP_DATE, a.MRR, a.SEGMENT,
         AVG(IFF(u.MONTH_START >= DATEADD(month, -3, c.AS_OF), u.LOGINS, NULL)) AS L3,
         AVG(IFF(u.MONTH_START < DATEADD(month, -3, c.AS_OF), u.LOGINS, NULL)) AS P3,
         AVG(IFF(u.MONTH_START >= DATEADD(month, -3, c.AS_OF), u.REPORTS_SENT, NULL)) AS R3,
         MAX(u.ACTIVE_USERS) AS SEATS
  FROM {DB}.RAW.ACCOUNTS a CROSS JOIN {DB}.FEATURES.SNAPSHOT_CALENDAR c
  LEFT JOIN {DB}.RAW.USAGE_EVENTS u
    ON u.ACCOUNT_ID = a.ACCOUNT_ID AND u.MONTH_START < c.AS_OF AND u.MONTH_START >= DATEADD(month, -6, c.AS_OF)
  WHERE a.SIGNUP_DATE < c.AS_OF
  GROUP BY 1, 2, 3, 4, 5)
SELECT ACCOUNT_ID, AS_OF,
       COALESCE(L3, 0)::FLOAT AS LOGINS_L3M,
       COALESCE(L3 / NULLIF(P3, 0), 1)::FLOAT AS LOGIN_TREND,
       COALESCE(R3, 0)::FLOAT AS REPORTS_L3M,
       COALESCE(SEATS, 1)::FLOAT AS ACTIVE_USERS,
       DATEDIFF(day, SIGNUP_DATE, AS_OF)::FLOAT AS TENURE_DAYS,
       MRR::FLOAT AS MRR,
       IFF(SEGMENT = 'TEAM', 1, 0)::FLOAT AS IS_TEAM
FROM agg""")

calls_df = session.sql(f"""
WITH sig AS (
  SELECT t.ACCOUNT_ID, t.CALL_TS, t.CALL_ID, t.SEGMENT_ID, t.TOPIC_SOURCE, t.TOPIC_ID, p.IS_NEGATIVE, p.PRICE_DRIVER
  FROM {DB}.ML.SEGMENT_TOPICS t LEFT JOIN {DB}.ANALYTICS.PRICE_SENTIMENT p USING (SEGMENT_ID))
SELECT a.ACCOUNT_ID, c.AS_OF,
       COUNT(DISTINCT s.CALL_ID)::FLOAT AS CALLS_90D,
       COUNT(s.SEGMENT_ID)::FLOAT AS SEGMENTS_90D,
       COUNT_IF(s.IS_NEGATIVE)::FLOAT AS PRICE_NEG_90D,
       COUNT_IF(s.PRICE_DRIVER = 'competitor price')::FLOAT AS PRICE_COMPETITOR_90D,
       COUNT_IF(s.PRICE_DRIVER = 'pricing model')::FLOAT AS PRICE_MODEL_90D,
       COUNT_IF(s.PRICE_DRIVER = 'value for money')::FLOAT AS PRICE_VALUE_90D,
       COUNT_IF(s.TOPIC_SOURCE = 'DISCOVERED')::FLOAT AS DISCOVERED_SEGMENTS_90D,
       COUNT_IF(s.TOPIC_ID = 'RENEWAL_TERMS')::FLOAT AS RENEWAL_SEGMENTS_90D
FROM {DB}.RAW.ACCOUNTS a CROSS JOIN {DB}.FEATURES.SNAPSHOT_CALENDAR c
LEFT JOIN sig s ON s.ACCOUNT_ID = a.ACCOUNT_ID AND s.CALL_TS < c.AS_OF AND s.CALL_TS >= DATEADD(day, -90, c.AS_OF)
GROUP BY 1, 2""")

usage_fv = FeatureView(name="ACCOUNT_USAGE_FV", entities=[account], feature_df=usage_df, timestamp_col="AS_OF",
                       refresh_freq="1 day", desc="Product usage per account at each snapshot date")
usage_fv = usage_fv.attach_feature_desc({
    "LOGINS_L3M": "Average monthly logins, last 3 months", "LOGIN_TREND": "Logins last 3 months over the 3 before",
    "REPORTS_L3M": "Average monthly reports sent, last 3 months", "ACTIVE_USERS": "Seats in use",
    "TENURE_DAYS": "Days since signup", "MRR": "Monthly recurring revenue", "IS_TEAM": "1 for team accounts"})
calls_fv = FeatureView(name="ACCOUNT_CALL_SIGNALS_FV", entities=[account], feature_df=calls_df, timestamp_col="AS_OF",
                       refresh_freq="1 day", desc="Call topics and price sentiment per account, 90 days before each date")
calls_fv = calls_fv.attach_feature_desc({
    "CALLS_90D": "Recorded conversations", "SEGMENTS_90D": "Conversation segments",
    "PRICE_NEG_90D": "Negative or mixed price remarks", "PRICE_COMPETITOR_90D": "Negative remarks driven by competitor price",
    "PRICE_MODEL_90D": "Negative remarks driven by the pricing model", "PRICE_VALUE_90D": "Negative remarks driven by value for money",
    "DISCOVERED_SEGMENTS_90D": "Segments in topics discovered by clustering", "RENEWAL_SEGMENTS_90D": "Segments about renewal terms"})
usage_fv = fs.register_feature_view(usage_fv, version="V1", block=True, overwrite=True)
calls_fv = fs.register_feature_view(calls_fv, version="V1", block=True, overwrite=True)
print(fs.list_feature_views().select("NAME", "VERSION", "REFRESH_FREQ", "SCHEDULING_STATE").to_pandas())

In [ ]:
pit = q(f"""
SELECT u.AS_OF::DATE::STRING AS AS_OF, ROUND(AVG(c.PRICE_NEG_90D), 3) AS AVG_PRICE_NEG, ROUND(AVG(u.LOGINS_L3M), 2) AS AVG_LOGINS
FROM {usage_fv.fully_qualified_name()} u JOIN {calls_fv.fully_qualified_name()} c USING (ACCOUNT_ID, AS_OF)
GROUP BY 1 ORDER BY 1""")
print(pit.to_string(index=False))
fvs = fs.list_feature_views().to_pandas()
ok = len(fvs) >= 2 and pit.AVG_PRICE_NEG.iloc[0] == 0 and pit.AVG_PRICE_NEG.iloc[-1] > 0 and pit.AVG_LOGINS.nunique() == len(pit)
check("S1", bool(ok), f"2 managed feature views over {len(pit)} snapshot dates; same definitions, different dates: "
      f"avg negative price remarks {pit.AVG_PRICE_NEG.tolist()}, avg logins {pit.AVG_LOGINS.tolist()}")

---
## 2. A point-in-time training set

The spine is every account at the snapshot date with its outcome over the following 270 days: `DURATION_DAYS` until it
cancelled (or 270 if it did not) and `EVENT` (1 = cancelled). The Feature Store joins each account's features as they were
**at** the snapshot, so nothing from January onward leaks in. The training set is cut twice from the same definitions: an
immutable, versioned **Dataset** (the snapshot you can hand to any trainer, inside Snowflake or out), and a table the
training procedure reads.

In [ ]:
spine = session.sql(f"""
SELECT ACCOUNT_ID, '{SNAPSHOT}'::TIMESTAMP_NTZ AS AS_OF,
       IFF(CHURN_DATE IS NULL, {HORIZON_DAYS}, DATEDIFF(day, '{SNAPSHOT}'::DATE, CHURN_DATE))::FLOAT AS DURATION_DAYS,
       IFF(CHURN_DATE IS NULL, 0, 1) AS EVENT,
       IFF({DB}.RAW.U(ACCOUNT_ID || 'split') < 0.7, 'TRAIN', 'TEST') AS SPLIT
FROM {DB}.RAW.SUBSCRIPTIONS""")
ds = fs.generate_dataset("CHURN_TRAINING", spine, [usage_fv, calls_fv], version="V1", spine_timestamp_col="AS_OF",
                         spine_label_cols=["DURATION_DAYS", "EVENT", "SPLIT"], desc="Churn training snapshot as of 2026-01-01")
fs.generate_training_set(spine, [usage_fv, calls_fv], save_as=f"{DB}.FEATURES.CHURN_TRAINING_SOURCE",
                         spine_timestamp_col="AS_OF", spine_label_cols=["DURATION_DAYS", "EVENT", "SPLIT"])
ds_rows = ds.read.to_snowpark_dataframe().count()
print(f"Dataset {ds.fully_qualified_name} V1: {ds_rows:,} rows; table FEATURES.CHURN_TRAINING_SOURCE written")

In [ ]:
src = q(f"""SELECT COUNT(*) AS N, AVG(EVENT) AS CHURN_RATE, COUNT_IF(SPLIT = 'TEST') AS TEST_N,
           MAX(PRICE_NEG_90D) AS MAX_PRICE_NEG, COUNT_IF(CALLS_90D > 0) AS WITH_CALLS
           FROM {DB}.FEATURES.CHURN_TRAINING_SOURCE""").iloc[0]
CHURN_RATE = float(src.CHURN_RATE)
check("S2", int(src.N) == ds_rows == 2000 and 0.15 < CHURN_RATE < 0.45,
      f"{int(src.N):,} accounts (test {int(src.TEST_N)}), churn within {HORIZON_DAYS} days {CHURN_RATE:.1%}, "
      f"{int(src.WITH_CALLS)} accounts with a recorded conversation before the snapshot; Dataset rows {ds_rows:,}")

---
## 3. Six models, trained next to the data

One stored procedure trains, on the same split:

| Model | What it predicts |
|---|---|
| Logistic regression | Will the account cancel within 270 days (yes/no)? Ignores when. |
| Cox proportional hazards (lifelines) | The risk of cancelling over time; each feature multiplies the hazard |
| Gradient-boosted survival (scikit-survival) | The same, with non-linear effects and interactions |

each on **usage features only** and on **usage + call features**. The comparison metric is the concordance index
(c-index) on the test accounts: the share of account pairs whose predicted risk is ordered the same way as their actual
time to cancel (0.5 = random). On ranking alone, logistic regression is a strong baseline; what the survival models add
is the probability of cancelling by **any** date from one model, and correct handling of accounts still active at the end
of follow-up (censored). The two survival models trained with call features are registered (Section 5).

In [ ]:
from snowflake.snowpark import Session

TRAIN_PACKAGES = ["snowflake-snowpark-python", "snowflake-ml-python==2.4.0", "lifelines==0.30.0", "scikit-survival==0.23.1",
                  "joblib", "numpy", "pandas"]


def train_churn_models(session: Session, horizon_days: int) -> str:
    import json as _j
    import joblib
    import numpy as _np
    import pandas as _pd
    from lifelines import CoxPHFitter
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sksurv.ensemble import GradientBoostingSurvivalAnalysis
    from sksurv.metrics import concordance_index_censored
    from sksurv.util import Surv
    from snowflake.ml.model import custom_model
    from snowflake.ml.registry import Registry

    db = "TRANSCRIPT_CHURN_ML_DEMO"
    usage = ["LOGINS_L3M", "LOGIN_TREND", "REPORTS_L3M", "ACTIVE_USERS", "TENURE_DAYS", "MRR", "IS_TEAM"]
    calls = ["CALLS_90D", "SEGMENTS_90D", "PRICE_NEG_90D", "PRICE_COMPETITOR_90D", "PRICE_MODEL_90D",
             "PRICE_VALUE_90D", "DISCOVERED_SEGMENTS_90D", "RENEWAL_SEGMENTS_90D"]
    src = session.table(f"{db}.FEATURES.CHURN_TRAINING_SOURCE")
    df = src.sort("ACCOUNT_ID").to_pandas()
    for c in usage + calls:
        df[c] = df[c].astype(float).fillna(1.0 if c == "LOGIN_TREND" else 0.0)
    tr, te = df[df.SPLIT == "TRAIN"], df[df.SPLIT == "TEST"]

    def cidx(frame, risk):
        return float(concordance_index_censored(frame.EVENT.astype(bool).values, frame.DURATION_DAYS.values, risk)[0])

    results, fitted = [], {}
    for fset_name, cols in (("usage", usage), ("usage+calls", usage + calls)):
        lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(tr[cols], tr.EVENT)
        results.append({"model": "logistic", "features": fset_name, "c_index": cidx(te, lr.predict_proba(te[cols])[:, 1])})
        cph = CoxPHFitter(penalizer=0.05).fit(tr[cols + ["DURATION_DAYS", "EVENT"]], "DURATION_DAYS", "EVENT")
        results.append({"model": "cox", "features": fset_name,
                        "c_index": cidx(te, cph.predict_partial_hazard(te[cols]).values)})
        gb = GradientBoostingSurvivalAnalysis(n_estimators=200, learning_rate=0.05, max_depth=3, subsample=0.8,
                                              random_state=0).fit(tr[cols].values,
                                                                  Surv.from_arrays(tr.EVENT.astype(bool), tr.DURATION_DAYS))
        results.append({"model": "gb_survival", "features": fset_name, "c_index": cidx(te, gb.predict(te[cols].values))})
        fitted[fset_name] = (cph, gb, cols)

    cph, gb, cols = fitted["usage+calls"]
    means = tr[cols].mean().to_dict()
    joblib.dump({"model": cph, "cols": cols, "means": means, "horizon": horizon_days}, "/tmp/cox.joblib")
    joblib.dump({"model": gb, "cols": cols, "horizon": horizon_days}, "/tmp/gb.joblib")

    def _prep(input_df, cols_):
        x = input_df[cols_].astype(float)
        return x.fillna({c: (1.0 if c == "LOGIN_TREND" else 0.0) for c in cols_})

    class CoxChurn(custom_model.CustomModel):
        """Cox model: churn probability within the horizon, and each feature's contribution to the log-hazard."""

        def __init__(self, context):
            super().__init__(context)
            import joblib as jl
            self.b = jl.load(self.context.path("bundle"))

        @custom_model.inference_api
        def predict(self, input_df: _pd.DataFrame) -> _pd.DataFrame:
            x = _prep(input_df, self.b["cols"])
            surv = self.b["model"].predict_survival_function(x, times=[self.b["horizon"]]).iloc[0].values
            return _pd.DataFrame({"CHURN_PROB": 1.0 - surv, "RISK_SCORE": self.b["model"].predict_log_partial_hazard(x).values})

        @custom_model.inference_api
        def explain(self, input_df: _pd.DataFrame) -> _pd.DataFrame:
            x = _prep(input_df, self.b["cols"])
            coef = self.b["model"].params_
            return _pd.DataFrame({f"{c}_CONTRIB": (x[c] - self.b["means"][c]) * float(coef[c]) for c in self.b["cols"]})

    class GbChurn(custom_model.CustomModel):
        """Gradient-boosted survival: churn probability within the horizon and a relative risk score."""

        def __init__(self, context):
            super().__init__(context)
            import joblib as jl
            self.b = jl.load(self.context.path("bundle"))

        @custom_model.inference_api
        def predict(self, input_df: _pd.DataFrame) -> _pd.DataFrame:
            x = _prep(input_df, self.b["cols"]).values
            fns = self.b["model"].predict_survival_function(x)
            return _pd.DataFrame({"CHURN_PROB": [1.0 - float(f(self.b["horizon"])) for f in fns],
                                  "RISK_SCORE": self.b["model"].predict(x)})

    sample = src.select(cols).limit(20)          # Snowpark sample: records lineage to the training table
    reg = Registry(session=session, database_name=db, schema_name="ML")
    scores = {(r["model"], r["features"]): r["c_index"] for r in results}
    for version, cls, path, deps, key in (
            ("COX_CALLS", CoxChurn, "/tmp/cox.joblib", ["lifelines==0.30.0", "joblib"], ("cox", "usage+calls")),
            ("GB_CALLS", GbChurn, "/tmp/gb.joblib", ["scikit-survival==0.23.1", "joblib"], ("gb_survival", "usage+calls"))):
        mv = reg.log_model(cls(custom_model.ModelContext(artifacts={"bundle": path})), model_name="CHURN_SURVIVAL",
                           version_name=version, sample_input_data=sample, conda_dependencies=deps,
                           target_platforms=["WAREHOUSE"],
                           comment=f"{key[0]} on usage + call features; trained on FEATURES.CHURN_TRAINING_SOURCE")
        mv.set_metric("test_c_index", round(scores[key], 4))
        mv.set_metric("horizon_days", horizon_days)
    best = "COX_CALLS" if scores[("cox", "usage+calls")] >= scores[("gb_survival", "usage+calls")] else "GB_CALLS"
    session.sql(f"ALTER MODEL {db}.ML.CHURN_SURVIVAL SET DEFAULT_VERSION = {best}").collect()
    return _j.dumps({"results": results, "default_version": best, "n_train": int(len(tr)), "n_test": int(len(te)),
                     "cox_coefficients": {k: round(float(v), 4) for k, v in cph.params_.items()}})


session.sql(f"CREATE STAGE IF NOT EXISTS {DB}.ML.CODE_STAGE").collect()
session.sproc.register(train_churn_models, name=f"{DB}.ML.TRAIN_CHURN_MODELS", is_permanent=True, replace=True,
                       stage_location=f"@{DB}.ML.CODE_STAGE", packages=TRAIN_PACKAGES, execute_as="caller",
                       comment="Trains logistic, Cox and gradient-boosted survival churn models; registers the survival ones")
print(f"Registered {DB}.ML.TRAIN_CHURN_MODELS(NUMBER) with {TRAIN_PACKAGES}")

In [ ]:
%%sql -r train_models
CALL TRANSCRIPT_CHURN_ML_DEMO.ML.TRAIN_CHURN_MODELS(270)

In [ ]:
TRAIN = json.loads(train_models.iloc[0, 0])
res = pd.DataFrame(TRAIN["results"])
print(res.pivot(index="model", columns="features", values="c_index").round(4))
best_usage = res[res.features == "usage"].c_index.max()
best_calls = res[res.features == "usage+calls"].c_index.max()
surv_calls = res[(res.features == "usage+calls") & (res.model != "logistic")].c_index.max()
logit_calls = float(res[(res.features == "usage+calls") & (res.model == "logistic")].c_index.iloc[0])
C_LIFT = best_calls - best_usage
SURV_VS_LOGIT = surv_calls - logit_calls
check("S3", C_LIFT >= 0.03 and abs(SURV_VS_LOGIT) <= 0.01,
      f"best c-index usage-only {best_usage:.4f} -> with call features {best_calls:.4f} (lift {C_LIFT:+.4f}, >= 0.03); "
      f"best survival with calls {surv_calls:.4f} vs logistic with calls {logit_calls:.4f} "
      f"({SURV_VS_LOGIT:+.4f}: same ranking, but only survival gives the probability at any horizon); "
      f"train {TRAIN['n_train']} / test {TRAIN['n_test']}")

---
## 4. Every candidate in one experiment

Each model and feature set is a run in the `CHURN_SURVIVAL` experiment with its parameters and test c-index, so the
comparison stays reproducible and visible in Snowsight (**AI & ML » Experiments**).

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(session=session, database_name=DB, schema_name="ML")
try:
    exp.delete_experiment("CHURN_SURVIVAL")
except Exception:
    pass
exp.set_experiment("CHURN_SURVIVAL")
for r in TRAIN["results"]:
    run_name = f"{r['model']}_{r['features'].replace('+', '_')}".upper()
    with exp.start_run(run_name):
        exp.log_params({"model": r["model"], "features": r["features"], "horizon_days": 270,
                        "n_features": len(ALL_FEATURES if r["features"] == "usage+calls" else USAGE_FEATURES),
                        "registered_version": {"cox": "COX_CALLS", "gb_survival": "GB_CALLS"}.get(r["model"], "")
                        if r["features"] == "usage+calls" else ""})
        exp.log_metrics({"test_c_index": round(r["c_index"], 4)})
print("logged", len(TRAIN["results"]), "runs to experiment CHURN_SURVIVAL")

In [ ]:
runs = session.sql(f"SHOW RUNS IN EXPERIMENT {DB}.ML.CHURN_SURVIVAL").collect()
check("S4", len(runs) == 6, f"experiment ML.CHURN_SURVIVAL has {len(runs)} runs: {sorted(r['name'] for r in runs)}")

---
## 5. The survival models in the registry

`ML.CHURN_SURVIVAL` holds both survival models trained with call features; the one with the higher test c-index is the
default. Each returns the probability of cancelling within 270 days. The Cox version also has an `explain` method: each
feature's contribution to the account's log-hazard relative to an average account, which is what a success manager needs
to hear ("negative remarks about the pricing model", not "risk 0.71").

In [ ]:
%%sql -r show_churn_versions
SHOW VERSIONS IN MODEL TRANSCRIPT_CHURN_ML_DEMO.ML.CHURN_SURVIVAL

In [ ]:
vers = {r.as_dict()["name"]: r.as_dict() for r in session.sql(f"SHOW VERSIONS IN MODEL {DB}.ML.CHURN_SURVIVAL").collect()}
default = next(n for n, v in vers.items() if str(v.get("is_default_version")).lower() == "true")
coef = TRAIN["cox_coefficients"]
top = sorted(coef.items(), key=lambda kv: -abs(kv[1]))[:4]
check("S5", {"COX_CALLS", "GB_CALLS"} <= set(vers) and default == TRAIN["default_version"],
      f"versions {sorted(vers)}, default {default}; largest Cox coefficients {top}")

---
## 6. Score from SQL, through the Feature Store, and monitor it

Scoring reads the same feature views at each date in the calendar (`retrieve_feature_values`), and the model runs where
the rows are, with `MODEL(...)!PREDICT` in SQL. `ANALYTICS.CHURN_SCORES` keeps one row per account per date with the
outcome over the following 270 days, so a **model monitor** tracks the score distribution, drift and AUC over time.

In [ ]:
score_spine = session.sql(f"SELECT a.ACCOUNT_ID, c.AS_OF FROM {DB}.RAW.ACCOUNTS a CROSS JOIN {DB}.FEATURES.SNAPSHOT_CALENDAR c "
                          f"WHERE a.SIGNUP_DATE < c.AS_OF")
fs.retrieve_feature_values(score_spine, [usage_fv, calls_fv], spine_timestamp_col="AS_OF") \
  .write.save_as_table(f"{DB}.FEATURES.SCORING_INPUT", mode="overwrite")
args = ", ".join(f"s.{c}" for c in ALL_FEATURES)
SCORE_SQL = f"""
CREATE OR REPLACE TABLE {DB}.ANALYTICS.CHURN_SCORES COMMENT = 'Churn probability per account per as-of date, scored in place' AS
SELECT s.ACCOUNT_ID, s.AS_OF::TIMESTAMP_NTZ AS SCORED_AT, {', '.join('s.' + c for c in ALL_FEATURES)},
       {DB}.ML.CHURN_SURVIVAL!PREDICT({args}):CHURN_PROB::FLOAT AS CHURN_PROB,
       IFF(b.CHURN_DATE IS NOT NULL AND b.CHURN_DATE >= s.AS_OF::DATE
           AND b.CHURN_DATE < DATEADD(day, {HORIZON_DAYS}, s.AS_OF::DATE), 1, 0) AS ACTUAL_CHURN
FROM {DB}.FEATURES.SCORING_INPUT s JOIN {DB}.RAW.SUBSCRIPTIONS b USING (ACCOUNT_ID)"""
t0 = time.time()
session.sql(SCORE_SQL).collect()
SCORE_SECONDS = time.time() - t0
print(f"scored {one(f'SELECT COUNT(*) FROM {DB}.ANALYTICS.CHURN_SCORES'):,} account-dates in place in {SCORE_SECONDS:.0f} s")

In [ ]:
%%sql -r scores_by_date
SELECT SCORED_AT::DATE AS AS_OF, COUNT(*) AS ACCOUNTS, ROUND(AVG(CHURN_PROB), 3) AS AVG_CHURN_PROB,
       ROUND(AVG(ACTUAL_CHURN), 3) AS ACTUAL_RATE
FROM TRANSCRIPT_CHURN_ML_DEMO.ANALYTICS.CHURN_SCORES GROUP BY 1 ORDER BY 1

In [ ]:
top = q(f"""SELECT ACCOUNT_ID, ROUND(CHURN_PROB, 3) AS CHURN_PROB FROM {DB}.ANALYTICS.CHURN_SCORES
           WHERE SCORED_AT = '{SNAPSHOT}' AND PRICE_NEG_90D > 0 ORDER BY CHURN_PROB DESC LIMIT 1""").iloc[0]
args = ", ".join(f"s.{c}" for c in ALL_FEATURES)
# explain is a table function: one row of *_CONTRIB columns per input row
contrib = q(f"""WITH m AS MODEL {DB}.ML.CHURN_SURVIVAL VERSION COX_CALLS
SELECT e.* FROM {DB}.FEATURES.SCORING_INPUT s, TABLE(m!EXPLAIN({args})) e
WHERE s.ACCOUNT_ID = '{top.ACCOUNT_ID}' AND s.AS_OF = '{SNAPSHOT}'""").iloc[0]
contrib = contrib.rename(lambda c: c.replace("_CONTRIB", "")).astype(float).sort_values(ascending=False)
TOP_DRIVERS = contrib.head(3).round(3).to_dict()
print(f"highest-risk account with price complaints: {top.ACCOUNT_ID}, churn probability {top.CHURN_PROB}")
print(contrib.round(3).to_string())

In [ ]:
# The scores come from the default version, so the monitor watches that version. A monitor needs the model task:
# the monitored output is a churn probability against a 0/1 outcome, i.e. binary classification.
session.sql(f"ALTER MODEL {DB}.ML.CHURN_SURVIVAL MODIFY VERSION {TRAIN['default_version']} "
            "SET TASK = 'TABULAR_BINARY_CLASSIFICATION'").collect()
session.sql(f"""
CREATE MODEL MONITOR {DB}.ML.CHURN_MONITOR WITH
  MODEL = {DB}.ML.CHURN_SURVIVAL
  VERSION = '{TRAIN["default_version"]}'
  FUNCTION = 'predict'
  SOURCE = {DB}.ANALYTICS.CHURN_SCORES
  WAREHOUSE = {WH}
  REFRESH_INTERVAL = '1 day'
  AGGREGATION_WINDOW = '1 day'
  TIMESTAMP_COLUMN = SCORED_AT
  PREDICTION_SCORE_COLUMNS = ('CHURN_PROB')
  ACTUAL_CLASS_COLUMNS = ('ACTUAL_CHURN')""").collect()
print(f"monitor ML.CHURN_MONITOR on CHURN_SURVIVAL version {TRAIN['default_version']}")

In [ ]:
monitors = [r.as_dict() for r in session.sql(f"SHOW MODEL MONITORS IN SCHEMA {DB}.ML").collect()]
sb = scores_by_date
last = sb.iloc[-1]
auc = q(f"""SELECT AVG(IFF(a.CHURN_PROB > b.CHURN_PROB, 1, IFF(a.CHURN_PROB = b.CHURN_PROB, 0.5, 0))) AS AUC
           FROM (SELECT * FROM {DB}.ANALYTICS.CHURN_SCORES WHERE SCORED_AT = '{SNAPSHOT}' AND ACTUAL_CHURN = 1) a
           CROSS JOIN (SELECT * FROM {DB}.ANALYTICS.CHURN_SCORES WHERE SCORED_AT = '{SNAPSHOT}' AND ACTUAL_CHURN = 0) b""").iloc[0, 0]
SNAPSHOT_AUC = float(auc)
check("S6", len(monitors) == 1 and len(sb) == 4 and int(last.ACCOUNTS) == 2000 and SNAPSHOT_AUC > 0.6,
      f"ANALYTICS.CHURN_SCORES {int(sb.ACCOUNTS.sum()):,} account-dates scored in SQL in {SCORE_SECONDS:.0f} s; "
      f"avg churn probability by date {sb.AVG_CHURN_PROB.tolist()}; AUC at {SNAPSHOT} {SNAPSHOT_AUC:.3f}; "
      f"monitor {monitors[0]['name']}; top-risk account drivers {TOP_DRIVERS}")

---
## Summary

Call topics and price sentiment are features in the same Feature Store as product usage, computed from raw tables at any
date. Survival models trained next to the data rank accounts better with the call features than without, are versioned
and explainable in the registry, score from SQL, and are monitored. Next: `05` measures what staying in place saves.